In [5]:
import pandas as pd
import numpy as np

# 1. CSV 데이터 불러오기 (이 부분이 빠져서 에러가 났던 것입니다!)
# 만약 파일명이 knhanes_like_week4 (1).csv 라면 아래 파일명도 똑같이 바꿔주세요.
df_raw = pd.read_csv('knhanes_like_week4 (1).csv')
df = df_raw.copy()

# 2. 2팀 노출 변수(과거흡연) 설정
df['exposure'] = (df['smk'] == 2).astype(int)

w = df['wt'].to_numpy()
ST = df['kstrata'].to_numpy()
PS = df['psu'].to_numpy()

# 3. 설계 기반 분산 및 도메인 평균 계산 함수
def _var_psu(z, strata, psu):
    t = pd.DataFrame({'st': strata, 'cl': psu, 'z': z})
    zc = t.groupby(['st', 'cl'])['z'].sum().reset_index()
    var = 0.0
    for s, g in zc.groupby('st'):
        m = len(g)
        if m > 1:
            var += m / (m - 1) * np.sum((g['z'] - g['z'].mean()) ** 2)
    return var

def svymean_domain(y, w, strata, psu, domain):
    y = np.asarray(y, float)
    d = np.asarray(domain, float) * (~np.isnan(y))
    y = np.nan_to_num(y) * d
    wd = np.asarray(w, float) * d
    th = np.sum(wd * y) / np.sum(wd)
    return th, np.sqrt(_var_psu(wd * (y - th) / np.sum(wd), strata, psu))

# 4. 팀 과제용 3개 열 도메인 정의
DOMS = {
    '전체': np.ones(len(df), bool),
    '과거흡연군 (노출)': df['exposure'] == 1,
    '비과거흡연군 (비노출)': df['exposure'] == 0
}

# 5. 행 생성 함수
def row_cont(label, var):
    """연속형: 가중 평균 (SE)"""
    r = {'특성': label, 'n': int(df[var].notna().sum())}
    for name, dom in DOMS.items():
        th, se = svymean_domain(df[var], w, ST, PS, dom)
        r[name] = f'{th:.1f} ({se:.2f})'
    return r

def row_cat(label, mask):
    """범주형: n (가중 %)"""
    mask = np.asarray(mask, float)
    r = {'특성': label, 'n': int(mask.sum())}
    for name, dom in DOMS.items():
        th, se = svymean_domain(mask, w, ST, PS, dom)
        r[name] = f'{th*100:.1f}'
    return r

# 6. Table 1 최종 생성 및 출력
t1 = pd.DataFrame([
    row_cont('연령 (세)', 'age'),
    row_cat('  65세 이상', df.age >= 65),
    row_cat('남성', df.sex == 1),
    row_cat('읍면 거주', df.urban == 2),
    row_cont('BMI (kg/m²)', 'bmi'),
    row_cat('  비만 (BMI≥25)', df.bmi >= 25),
    row_cont('수축기혈압 (mmHg)', 'sbp'),
    row_cat('고혈압', df.htn == 1),
    row_cat('규칙적 운동', df.exercise == 2),
])

t1

,특성,n,전체,과거흡연군 (노출),비과거흡연군 (비노출)
0,연령 (세),5367,49.1 (0.16),50.0 (0.53),49.0 (0.19)
1,65세 이상,1607,16.8,18.0,16.6
2,남성,2719,49.0,40.8,50.6
3,읍면 거주,2156,22.9,24.1,22.7
4,BMI (kg/m²),5367,23.8 (0.05),24.0 (0.14),23.8 (0.06)
5,비만 (BMI≥25),2035,36.5,39.3,36.0
6,수축기혈압 (mmHg),5367,122.8 (0.45),123.7 (0.71),122.6 (0.46)
7,고혈압,1652,24.4,25.5,24.2
8,규칙적 운동,2238,41.5,42.4,41.3
